# Proyecto 2 FYHBD | Persona 1 — Capa Bronze (`bronze_ingest.ipynb`)

Ingesta **manual de respaldo** (si falla Airflow). Usa exactamente la misma lógica que `scripts/bronze_ingest.py`.

```
Fuente (S3 público ClickHouse, StackOverflow) → dlt (PyArrow por lotes) → Parquet → MinIO / bucket `bronze`
```

**Bronze = datos crudos en Parquet, con sobrescritura permitida** (`write_disposition="replace"`): re-ejecutar no duplica.

| Dataset / año | Quién lo carga | Ruta en MinIO |
|---|---|---|
| post 2020, post 2021 | manual | `s3://bronze/post/<año>/post_<año>.parquet` |
| users 2020 | manual | `s3://bronze/users/2020/users_2020.parquet` |
| **users 2021** | **pipeline Airflow** (o este notebook) | `s3://bronze/users/2021/users_2021.parquet` |

> `users` está publicado como **un solo archivo** (`users.parquet`); el script filtra por año de `CreationDate`.

## 1. Parámetros

In [ ]:
DATASET = "users"          # post | users | votes | comments | badges | postlinks
YEAR    = 2021
URL     = None             # None = fuente pública; o una ruta/URL alterna
CARGAR_MANUAL = False      # True = además carga post 2020, post 2021 y users 2020 (tarda: posts es pesado)

## 2. Importar el módulo de ingesta

In [ ]:
import sys
sys.path.insert(0, "/workspace/scripts")   # en el contenedor Jupyter
sys.path.insert(0, "../scripts")           # por si se corre fuera de Docker
import bronze_ingest as b

print("Buckets/endpoint:", b._minio_cfg()["endpoint"], "| bucket:", b.BUCKET)
print("Fuente:", b.source_url(DATASET, YEAR, URL))
print("Destino:", "s3://" + b.target_path(DATASET, YEAR))

## 3. Ingesta del dataset/año indicado (equivale a la tarea Bronze de Airflow)

In [ ]:
resumen = b.run_bronze(DATASET, YEAR, URL)
resumen

## 4. (Opcional) Carga manual previa: post 2020, post 2021, users 2020

In [ ]:
if CARGAR_MANUAL:
    for ds, y in b.MANUAL_LOADS:
        b.run_bronze(ds, y)
else:
    print("CARGAR_MANUAL=False: se omite. Cambie el parámetro a True para cargarlos.")

## 5. Verificación: contenido de Bronze en MinIO

In [ ]:
b.listar_bronze()

## 6. Verificación del esquema (columnas originales, sin renombrar)

In [ ]:
import pyarrow.parquet as pq
fs = b.minio_fs()
with fs.open(b.target_path(DATASET, YEAR), "rb") as fh:
    pf = pq.ParquetFile(fh)
    print(pf.metadata.num_rows, "filas |", pf.metadata.num_columns, "columnas")
    print(pf.schema_arrow)